# Cycle 6 : Test undersampling


In [12]:
### Import des modules
%load_ext autoreload
%autoreload 2

import pandas as pd
import utils
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
data = pd.read_csv('data/rafined/employees.csv', sep=',')

In [14]:
train_data = utils.split_train_data(data, 'a_quitte_l_entreprise')

In [15]:
preprocessor = utils.create_preprocessor()

In [16]:
from sklearn.feature_selection import SelectFromModel
import numpy as np

feature_selector = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    max_features=20,
    threshold=-np.inf
)

final_model = RandomForestClassifier(
    max_depth=6,
    min_samples_leaf=15,
    random_state=42
)

In [18]:
from technova_correlation_cleaning import CorrelationFilter
from sklearn.linear_model import LogisticRegression
from technova_features import TechNovaFeatureEngineering
from imblearn.under_sampling import NearMiss
from imblearn.pipeline import Pipeline

random_forest_model = RandomForestClassifier(random_state=42)
logistic_regression_model = LogisticRegression(random_state=42, max_iter=1000)

models = [
    {'name' : 'LogisticRegression', 'model': logistic_regression_model},
    {'name' : 'RandomForestClassifier', 'model': random_forest_model},
]

for model in models:
    print(f'Training model: {model["name"]}')

    pipeline = Pipeline([
        ('features', TechNovaFeatureEngineering()),
        ('corr_cleaning', CorrelationFilter(threshold=0.80)),
        ('preprocessor', preprocessor),
        ('feature_selection', feature_selector),
        ('sampling', NearMiss(version=1)),
        ('model', model['model']),
    ])

    utils.benchmark(pipeline, train_data)

    print(f'------------------------------\n')

Training model: LogisticRegression
--- Validation Fold Results ---
Validation Recall : [0.78947368 0.84210526 0.73684211 0.63157895 0.78947368], Recall moyen : 0.7578947368421053
Validation F1-Scores [0.33707865 0.35555556 0.31638418 0.28571429 0.33149171], F1 moyen : 0.32524487729067547
Validation ROC AUC : [0.60871877 0.64293348 0.586428   0.59350788 0.62530056], ROC moyen : 0.6113777367584474

--- Train Fold Results (Overfit Check) ---
Train Recall : [0.81578947 0.77631579 0.80921053 0.84210526 0.82894737], Recall moyen : 0.8144736842105263
Train F1-Scores [0.36098981 0.32196453 0.33744856 0.35955056 0.33333333], F1 moyen : 0.3426573589809705
Train ROC AUC : [0.68072903 0.6052715  0.6161697  0.66496565 0.59818391], ROC moyen : 0.633063956394141
------------------------------

Training model: RandomForestClassifier
--- Validation Fold Results ---
Validation Recall : [0.81578947 0.89473684 0.68421053 0.81578947 0.78947368], Recall moyen : 0.8
Validation F1-Scores [0.30693069 0.3300970